# Llama: сравнение transfer pruning и Torch-Pruning

Первый, средний и последний слои сравниваются в одном запуске через `ComparePrunedModelsUsecase`.
Сравниваем активации, без градиентов. Usecase структурно изменяет модель: перед повторным запуском сравнения загрузите её заново.


In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pruning_lens").is_dir())
sys.path.insert(0, str(ROOT))

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformer_lens.model_bridge import TransformerBridge
from lora_transfer_pruning.core.prune_task_type import GroupPruneTask, ModelPruneTask
from pruning_lens.usecase.compare_pruned_models_usecase import ComparePrunedModelsUsecase

MODEL = "meta-llama/Llama-3.1-8B-Instruct"
DEVICE = "cuda:2"
DTYPE = torch.bfloat16

model = AutoModelForCausalLM.from_pretrained(
    MODEL, trust_remote_code=False, device_map={"": DEVICE},
    dtype=DTYPE, attn_implementation="eager",
).eval()
bridge = TransformerBridge.boot_transformers(MODEL, hf_model=model, dtype=DTYPE)


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=False)
tokens = tokenizer(
    "The attention mechanism allows a language model to use information from earlier tokens.",
    return_tensors="pt",
).input_ids.to(DEVICE)

In [3]:
n_layers = len(bridge.blocks)
layers = [0, n_layers // 2, n_layers - 1]
prune_task = ModelPruneTask({
    f"blocks.{layer}.attn.q_proj": GroupPruneTask(cols=None, rows=[1, 2])
    for layer in layers
})

# bridge.set_use_hook_mlp_in(True)
# bridge.set_use_attn_result(True)
# bridge.set_use_attn_in(True)

prefixes = tuple(f"blocks.{layer}." for layer in layers)
#if we dont use use_split_qkv in PositionEmbeddingsAttentionBridge
# excluded_postfixes = (".hook_attn_in", ".hook_mlp_in", ".hook_q_input", ".hook_k_input", ".hook_v_input", ".hook_result")
#оставить exclude как переключалку в методе без on layers
name_filter = lambda name: name.startswith(prefixes) #and not name.endswith(excluded_postfixes)

compairing_trace, sim_prune_trace, tp_trace = ComparePrunedModelsUsecase.compare_transfer_and_torch_prunings_on_layers(
    bridge, tokens, prune_task, layers=layers, compute_gradient=True,
)


/glazkov-dev/LoRa-Transfer-Pruning/.venv/lib/python3.10/site-packages/torch_pruning/dependency/graph.py:390: UserWarning: Unwrapped parameters detected: ['model.layers.0._original_component.input_layernorm._original_component.weight', 'model.layers.3._original_component.input_layernorm._original_component.weight', 'model.layers.6._original_component.input_layernorm._original_component.weight', 'model.layers.25._original_component.input_layernorm._original_component.weight', 'model.layers.28._original_component.post_attention_layernorm._original_component.weight', 'model.layers.3._original_component.post_attention_layernorm._original_component.weight', 'model.layers.10._original_component.post_attention_layernorm._original_component.weight', 'model.layers.18._original_component.input_layernorm._original_component.weight', 'model.layers.22._original_component.post_attention_layernorm._original_component.weight', 'model.layers.31._original_component.post_attention_layernorm._original_comp

[00] blocks.0.attn.q.hook_in
     FWD    OK            shape=(1, 15, 4096) compared=61440 max_abs=0.000e+00 mean_abs=0.000e+00 rmse=0.000e+00 rel_l2=0.000e+00 abs_l2=0.000e+00
     BWD    DIFF          shape=(1, 15, 4096) compared=61440 max_abs=7.324e-04 mean_abs=1.617e-05 rmse=4.793e-05 rel_l2=2.192e-02 abs_l2=1.188e-02
[01] blocks.0.attn.q.hook_out
     FWD    OK            shape=(1, 15, 4096) compared=61440 max_abs=0.000e+00 mean_abs=0.000e+00 rmse=0.000e+00 rel_l2=0.000e+00 abs_l2=0.000e+00
     BWD    DIFF          shape=(1, 15, 4096) compared=61440 max_abs=4.883e-04 mean_abs=3.665e-06 rmse=1.608e-05 rel_l2=1.963e-02 abs_l2=3.986e-03
[02] blocks.0.attn.hook_q
     FWD    SHAPE         shape=(1, 15, 3968) compared=0 candidate_shape=(1, 15, 4096)
     BWD    SHAPE         shape=(1, 15, 3968) compared=0 candidate_shape=(1, 15, 4096)
[03] blocks.0.attn.k.hook_in
     FWD    OK            shape=(1, 15, 4096) compared=61440 max_abs=0.000e+00 mean_abs=0.000e+00 rmse=0.000e+00 rel_l2=0.00

In [4]:
compairing_trace.activations['blocks.0.attn.o.hook_in']

TensorComparison(status=<ComparisonStatus.OK: 'OK'>, difference=tensor([[[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]]]), max_abs=0.0, mean_abs=0.0, rmse=0.0, rel_l2=0.0, abs_l2=0.0, reference_shape=(1, 15, 3968), candidate_shape=(1, 15, 32, 128), reference_restored_shape=(1, 15, 4096), candidate_restored_shape=(1, 15, 4096), compared_elements=61440, reason=None)

In [ ]:
compairing_trace.gradients['blocks.0.attn.q.hook_in'].difference

tensor([[[ True,  True,  True,  ...,  True,  True,  True],
         [False, False, False,  ..., False, False, False],
         [False, False, False,  ..., False, False, False],
         ...,
         [False, False, False,  ...,  True, False, False],
         [False, False, False,  ..., False, False, False],
         [ True,  True,  True,  ...,  True,  True,  True]]])

In [16]:
compairing_trace.print_summary()

[00] blocks.0.attn.q.hook_in
     FWD    OK            shape=(1, 15, 4096) compared=61440 max_abs=0.000e+00 mean_abs=0.000e+00 rmse=0.000e+00 rel_l2=0.000e+00 abs_l2=0.000e+00
     BWD    DIFF          shape=(1, 15, 4096) compared=61440 max_abs=7.324e-04 mean_abs=1.617e-05 rmse=4.793e-05 rel_l2=2.192e-02 abs_l2=1.188e-02
[01] blocks.0.attn.q.hook_out
     FWD    OK            shape=(1, 15, 4096) compared=61440 max_abs=0.000e+00 mean_abs=0.000e+00 rmse=0.000e+00 rel_l2=0.000e+00 abs_l2=0.000e+00
     BWD    DIFF          shape=(1, 15, 4096) compared=61440 max_abs=4.883e-04 mean_abs=3.665e-06 rmse=1.608e-05 rel_l2=1.963e-02 abs_l2=3.986e-03
[02] blocks.0.attn.hook_q
     FWD    SHAPE         shape=(1, 15, 3968) compared=0 candidate_shape=(1, 15, 4096)
     BWD    SHAPE         shape=(1, 15, 3968) compared=0 candidate_shape=(1, 15, 4096)
[03] blocks.0.attn.k.hook_in
     FWD    OK            shape=(1, 15, 4096) compared=61440 max_abs=0.000e+00 mean_abs=0.000e+00 rmse=0.000e+00 rel_l2=0.00

Расхождение в o hook out подтверждается прошлым файлом

LoRa-Transfer-Pruning/experiments/compare_tp_and_our/compare_tp_and_ours_llama_and_gemm_problem.ipynb